**New benchmark analysis (2026-10-05).** Uses the available 4L/2H/K16 checkpoints with weight decay **0.1**, seeds 100–104. This is a new exploratory analysis, not reproduction of SEL-4L wd0.01. References/conflict definitions are frozen historical inputs, verified by hash. All model-dependent outputs are recomputed. Validation match splits support the stated comparisons; prior test access prevents a fresh confirmatory claim. Population and intervention scope apply to each reported result. See `config/analysis_plan.json` and `SURGICAL_FIXES.md`.

# MB13 — Which Context Input Is Doing the Work? (side study)

**Track B · Mechanistic interpretability · Identity × Context project**

| Item | Content |
|---|---|
| **Question** | "Context" has been handled as one thing everywhere else in Track B. It is not one thing: it is fourteen separate model inputs. Which of them actually move the arbitration, and by how much relative to each other? |
| **Purpose** | A breadth-first survey, not a mechanism. MB04 and MB06 change context by swapping a whole donor window, which moves every context input at once — good for asking *whether* context interacts with identity, useless for asking *which part of context* does. This notebook puts a number next to each input so later work knows which ones are worth a proper experiment. |
| **Inputs** | `artifacts/populations/*`, `config/preregistration.json`, SEL-4L primary seed |
| **Outputs** | `results/MB13_*`, `figures/MB13_*` |
| **Model under study** | SEL-4L, primary seed |
| **Test data** | none |
| **Status** | **Side study.** Not pre-registered, not part of the MB12 lock, not part of the circuit. Nothing here may be cited as confirmatory. |

**Deliberately shallow.** Every input gets the same simple treatment and the same few numbers. There
is no per-input circuit, no path patching, no held-out confirmation. The point is a ranked map of
the context side, so that the one remaining open question from the audit — *which* context variable
gates identity — has somewhere sensible to start. Run any time after MB06.

> **⚠ REPAIR STATUS (2026-10-04).** The review found that several of this survey's interventions do not match their names,
> and that its significance column cannot rank inputs. Findings, all confirmed against the code:
> **F14** marginal draws were called "on-distribution"; **F15** the resampler could keep the original value (expected unchanged
> share: same_player_as_prev 44%, possession_flag 32%) and restarted the same random stream every batch; **F16** "adjacent"
> was ±1 on code numbers, including unordered categories and special codes (OTHER → END), always +1 except at the top code;
> **F17** "group removed" set context codes to 0, which selects a learned category, not removal; **F18** |joint| − Σ|single|
> mixes different endpoints and cancels signs, so it cannot show additivity or interaction; **F19** the whole-window swap is no
> upper bound and the random vector no floor; **F20** every |ΔΔ| > 0 test hits the bootstrap floor (all q = 0.0005), so the q
> column carries no ranking information; **F21** restoring the whole target residual gives R = 1 at every depth by
> construction. **F11/F12:** results are about P-player conflict rows (1,744), selected partly for confident context
> references, and do not generalise to all events.
>
> The defective legacy survey sections are disabled in this new benchmark. **Section R** re-runs the survey
> with interventions that match their names, and drops the significance column in favour of effect sizes with intervals. It
> still has no discovery/estimation split and nothing in it is confirmatory.

In [1]:
import os
for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_var] = os.environ.get("MI_CPU_THREADS", "4")
import sys
import json
import time
import importlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_root(start):
    """`start` (or one of its ancestors) holding both project folders directly, else None."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training_evaluation").is_dir() and (candidate / "mi_analysis").is_dir():
            return candidate
    return None


def search_root(bases, max_depth=6):
    """Breadth-first search under each base (e.g. a mounted Drive) for the project root."""
    for base in bases:
        base = Path(base)
        if not base.is_dir():
            continue
        found = find_root(base)
        if found is not None:
            return found
        frontier = [(base, 0)]
        while frontier:
            folder, depth = frontier.pop(0)
            if depth >= max_depth:
                continue
            try:
                children = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".")]
            except PermissionError:
                continue
            for child in children:
                if child.name == "model_training_evaluation" and (child.parent / "mi_analysis").is_dir():
                    return child.parent
                frontier.append((child, depth + 1))
    return None


if "SOCCER_MI_ROOT" in os.environ:
    ROOT = Path(os.environ["SOCCER_MI_ROOT"])
elif IN_COLAB:
    ROOT = search_root(["/content", "/content/drive/MyDrive", "/content/drive/My Drive"])
    if ROOT is None:
        raise RuntimeError(
            "BLOCKED: could not find a folder holding both model_training_evaluation/ and mi_analysis/ "
            "under /content or the mounted Drive. Upload or sync the project there, or set "
            "SOCCER_MI_ROOT to its exact path in a cell ABOVE this one, for example: "
            "import os; os.environ['SOCCER_MI_ROOT'] = '/content/drive/MyDrive/.../soccer_mi'")
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise RuntimeError(f"BLOCKED: no ancestor of {Path.cwd()} holds both model_training_evaluation/ "
                           "and mi_analysis/. Run this notebook from inside the project, or set SOCCER_MI_ROOT.")
TRACK_A_DIR = ROOT / "model_training_evaluation"
MI_DIR = Path(os.environ.get("MI_ANALYSIS_DIR", ROOT / "mi_analysis")).resolve()
assert (TRACK_A_DIR / "src" / "data.py").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'src' / 'data.py'} not found -- ROOT resolved to {ROOT}, which is wrong. "
    "Set SOCCER_MI_ROOT explicitly.")
assert (TRACK_A_DIR / "artifacts" / "events_prepared.parquet").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'artifacts' / 'events_prepared.parquet'} not found -- ROOT is wrong.")
RUN_MODE = os.environ.get("MI_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

for folder in [MI_DIR, MI_DIR / "src", MI_DIR / "notebooks", MI_DIR / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
(MI_DIR / "src" / "__init__.py").touch()
os.chdir(MI_DIR)

sys.path[:] = [entry for entry in sys.path if entry not in ("", ".", str(MI_DIR))]
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))
spec = importlib.util.spec_from_file_location("mi_src", MI_DIR / "src" / "__init__.py",
                                              submodule_search_locations=[str(MI_DIR / "src")])
mi_src = importlib.util.module_from_spec(spec)
sys.modules["mi_src"] = mi_src
spec.loader.exec_module(mi_src)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("ROOT:", ROOT)
print("Track A (read only):", TRACK_A_DIR, "| exists:", TRACK_A_DIR.exists())
print("Track B:", MI_DIR)

Colab: True | RUN_MODE: full
ROOT: /content/soccer_mi
Track A (read only): /content/soccer_mi/model_training_evaluation | exists: True
Track B: /content/soccer_mi/.mi_analysis_rerun


In [2]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.metrics as metrics
import src.training as training
import mi_src.paths as paths
import mi_src.fingerprint as fingerprint
import mi_src.mi_setup as mi_setup
import mi_src.hooks as hooks
import mi_src.interventions as interventions
import mi_src.mi_metrics as mi_metrics
import mi_src.patching as patching
import mi_src.circuits as circuits
import mi_src.attribution as attribution
for module in [data, metrics, training, paths, fingerprint, mi_setup, hooks, interventions, mi_metrics, patching, circuits, attribution]:
    importlib.reload(module)

if getattr(paths, "SCHEMA_VERSION", 1) < 2:
    raise RuntimeError("BLOCKED: mi_analysis/src/paths.py is older than this notebook expects. "
                       "Re-run MB01 section 0 (its three %%writefile cells), then restart the kernel.")
paths.check_modules(**{name: module for name, module in
                       [("paths", paths), ("fingerprint", fingerprint), ("mi_setup", mi_setup),
                        ("patching", globals().get("patching")), ("circuits", globals().get("circuits"))]
                       if module is not None})

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUT = paths.output_dir(MI_DIR, RUN_MODE)
SETTINGS = paths.RUN_SETTINGS[RUN_MODE]
TRACK = paths.track_a_paths(TRACK_A_DIR)
MANIFEST = OUT / "manifests" / "sha_manifest.json"
REGISTRY = OUT / "manifests" / "experiment_registry.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(int(os.environ.get("MI_CPU_THREADS", "4")))
if RUN_MODE == "full" and DEVICE != "cuda":
    print("Note: no GPU found. The full run works on CPU but is much slower.")
training.configure_determinism()
training.set_seed(0)
ACTIONS = list(data.ACTION_NAMES)
print("Device:", DEVICE, "| outputs:", OUT)
print("Settings:", SETTINGS)

Device: cuda | outputs: /content/soccer_mi/.mi_analysis_rerun
Settings: {'reference_epochs': 25, 'batch_windows': 256, 'n_bootstrap': 2000, 'n_draws': 5, 'n_permutations': 1000, 'n_random_circuits': 100, 'probe_train_events': 20000, 'max_events': 100000, 'probe_seeds': 3, 'scrub_draws': 5}


In [3]:
from src.model import FACTOR_GROUPS, CONTINUOUS_GROUPS

verified = fingerprint.verify_files(MANIFEST)
prereg = mi_setup.load_json(OUT / "config" / "preregistration.json")
mi_config = mi_setup.load_json(OUT / "config" / "mi_config.json")
IDENTITY = mi_config["identity_variable"]
PRIMARY_SEED = int(prereg["model"]["primary_seed"])


def run_dir(seed):
    """Run directory of one SEL-4L seed, re-rooted onto this checkout."""
    return paths.rebase_run_dir(prereg["model"]["run_dirs"][str(seed)], TRACK["root"])

N_BOOT = SETTINGS["n_bootstrap"]
BATCH = SETTINGS["batch_windows"]
DRAWS = SETTINGS["n_draws"]

events = pd.read_parquet(TRACK["events"])
bundle = mi_setup.build_bundle(events, TRACK, DEVICE)
val_view = bundle["views"]["val"]
POP = OUT / "artifacts" / "populations"
val_conflicts = pd.read_parquet(POP / "val_conflicts.parquet")
donors = pd.read_parquet(POP / "context_donors.parquet")
MODEL = mi_setup.load_model(run_dir(PRIMARY_SEED), bundle)
N_LAYERS = len(MODEL.blocks)
features = {name: tensor.cpu().numpy() for name, tensor in bundle["encoded"].tensors.items()}

CONTEXT_GROUPS = [group for group in FACTOR_GROUPS if group not in ("player", "team", "role", "competition")]
CATEGORICAL = [(group, name) for group in CONTEXT_GROUPS for name in FACTOR_GROUPS[group] if name in features]
CONTINUOUS = [(group, name) for group in CONTEXT_GROUPS for name in CONTINUOUS_GROUPS.get(group, []) if name in features]
population = val_conflicts[val_conflicts.population == "P-player"].merge(
    donors[["event_row", "donor_window"]], on="event_row", how="left").reset_index(drop=True)
print(f"verified files: {len(verified)} | primary seed {PRIMARY_SEED}")
print(f"context groups: {CONTEXT_GROUPS}")
print(f"categorical context inputs ({len(CATEGORICAL)}): {[n for _, n in CATEGORICAL]}")
print(f"continuous context inputs ({len(CONTINUOUS)}): {[n for _, n in CONTINUOUS]}")
print(f"population: {len(population):,} P-player conflict rows")

/content/soccer_mi/model_training_evaluation/src/batching.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)


verified files: 77 | primary seed 103
context groups: ['space', 'time', 'state', 'history']
categorical context inputs (10): ['zone', 'period', 'minute_bin', 'score_diff', 'poss_len', 'possession_flag', 'prev_action', 'prev_outcome', 'prev_end_zone', 'same_player_as_prev']
continuous context inputs (4): ['x_norm', 'y_norm', 'dt', 't_poss']
population: 1,744 P-player conflict rows


## 1. Terms

| Term | Meaning here |
|---|---|
| **Context input** | One of the model's own input features belonging to a non-identity factor group: space, time, state or history. The groups and their members are read from the model's code, not typed out here, so this notebook cannot drift from the architecture. |
| **Resample edit** | The value at the target is replaced by a value drawn from how that input is actually distributed over the conflict population, `n_draws` times, ~~never the event's own value. On-distribution by construction.~~ *[Corrected 2026-10-04 (F14, F15): a draw from the field's marginal while every other field stays fixed is not on-distribution (the joint combination can be unseen), and the code redraws only once, so the original value can remain.]* |
| **Adjacent edit** | The value at the target is moved one step up or down, where the input has more than two levels. ~~A smaller and more realistic change than a resample, and the one the blueprint prefers for ordered inputs (§22.2).~~ *[Corrected 2026-10-04 (F16): the code moves the integer code by +1 (−1 only at the top code) for every field with three or more codes, including unordered categories and special codes; it is a code remapping, not a local move.]* |
| **Shift edit** | For a continuous input: plus or minus one population standard deviation. |
| **Whole-context swap** | The MB06 context contrast — the donor window with the event's own player written back in. Every context input moves at once, ~~so it is the natural **upper bound** for any single input~~. *[Corrected 2026-10-04 (F19, F13): a joint change need not exceed a single change (effects can cancel), and the donor window also changes the history and the donor's identity fields. It is a reference perturbation, not a bound.]* |
| **Random-vector ~~floor~~ reference** *(F19: not a lower bound)* | The group's factor vector replaced by a random direction of the same length. What a change of that size does when it carries no learned information. |

Every edit acts at the **target position only** and asserts, field by field, that nothing else moved.

## 2. The inventory

Before any intervention: how many distinct values each context input takes on the conflict
population, and how concentrated it is. An input that is almost constant here cannot explain much
about *these* events whatever the model does with it in general, and its row below should be read
with that in mind.

In [4]:
rows = []
event_rows = population["event_row"].to_numpy()
for group, name in CATEGORICAL:
    values = features[name][event_rows]
    counts = pd.Series(values).value_counts(normalize=True)
    rows.append({"group": group, "input": name, "kind": "categorical", "distinct values": int(len(counts)),
                 "vocabulary size": int(features[name].max()) + 1,
                 "most common share": float(counts.iloc[0]), "spread (SD)": float(np.std(values))})
for group, name in CONTINUOUS:
    values = features[name][event_rows].astype(np.float64)
    rows.append({"group": group, "input": name, "kind": "continuous", "distinct values": int(len(np.unique(values))),
                 "vocabulary size": np.nan, "most common share": np.nan, "spread (SD)": float(values.std())})
inventory = pd.DataFrame(rows)
inventory.to_csv(OUT / "results" / "MB13_inventory.csv", index=False)
print(inventory.round(3).to_string(index=False))

  group               input        kind  distinct values  vocabulary size  most common share  spread (SD)
  space                zone categorical               96             96.0              0.063       27.818
   time              period categorical                2              5.0              0.525        0.499
   time          minute_bin categorical               20             24.0              0.074        5.374
  state          score_diff categorical                7              7.0              0.386        1.124
  state            poss_len categorical               21             22.0              0.131        6.703
  state     possession_flag categorical                2              2.0              0.649        0.477
history         prev_action categorical               15             16.0              0.168        4.440
history        prev_outcome categorical                3              3.0              0.446        0.677
history       prev_end_zone categorical       

## 3. One edit per input

Each input is edited on its own at the target position and the whole output distribution is
re-read. Four numbers per input: how far the arbitration moved (|ΔΔ|), how far the whole
distribution moved (KL over all 15 actions), and which way the two candidate actions went.

Reference rows are measured in the same pass:

* **no-op** — must be exactly 0, a bug detector.
* **whole-context swap** — the MB06 contrast, every context input at once; ~~the upper bound~~ *a reference perturbation (F19)*.
* **random vector (per group)** — the group's factor vector replaced by a random direction of the
  same norm; ~~the floor for anything in that group~~ *a reference perturbation, not a floor (F19)*.

Benjamini–Hochberg at q = 0.05 over the inputs, and a match-level bootstrap interval on every
mean. Both edit kinds are reported separately, because an adjacent step and a free resample are
different-sized questions.

> **⚠ REPAIR NOTE (2026-10-04) — INVALIDATED — INTERVENTION DESIGN — F14, F15, F16.** `resample_edit` redraws once (unchanged values remain) and reseeds per batch; `adjacent_edit` is ±1 on codes, including unordered and special categories. Kept to reproduce the historical file. Corrected: Section R2.

In [5]:
@torch.no_grad()
def measure(edit_name, make_inputs, scope_features, n_repeats=1):
    """Per-event effect of one edit: |change in Delta|, KL, and the move in each candidate action."""
    collected = {key: [] for key in ["abs_delta_change", "kl", "dlogp_ctx", "dlogp_id", "match_id"]}
    for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, population, BATCH):
        chunk = population.iloc[part]
        base = mi_metrics.at_target(hooks.forward(MODEL, inputs)[0], positions)
        base_np, a_ctx_np, a_id_np = base.cpu().numpy(), a_ctx.cpu().numpy(), a_id.cpu().numpy()
        base_log = mi_metrics.log_softmax(base_np)
        index = np.arange(len(base_np))
        totals = {key: np.zeros(len(chunk)) for key in ["abs_delta_change", "kl", "dlogp_ctx", "dlogp_id"]}
        for repeat in range(n_repeats):
            changed, replace = make_inputs(inputs, positions, chunk, repeat)
            if scope_features is not None:
                interventions.assert_scope(inputs, changed, scope_features, positions)
            new = mi_metrics.at_target(hooks.forward(MODEL, changed, replace=replace)[0], positions)
            new_np = new.cpu().numpy()
            new_log = mi_metrics.log_softmax(new_np)
            totals["abs_delta_change"] += np.abs(mi_metrics.delta(new_np, a_ctx_np, a_id_np)
                                                 - mi_metrics.delta(base_np, a_ctx_np, a_id_np)) / n_repeats
            totals["kl"] += mi_metrics.kl_divergence(base_np, new_np) / n_repeats
            totals["dlogp_ctx"] += (new_log[index, a_ctx_np] - base_log[index, a_ctx_np]) / n_repeats
            totals["dlogp_id"] += (new_log[index, a_id_np] - base_log[index, a_id_np]) / n_repeats
        for key, values in totals.items():
            collected[key].append(values)
        collected["match_id"].append(chunk["match_id"].to_numpy())
    frame = pd.DataFrame({key: np.concatenate(values) for key, values in collected.items()})
    frame["edit"] = edit_name
    return frame


def resample_edit(name, rng_seed):
    pool = features[name][population["event_row"].to_numpy()]
    def make(inputs, positions, chunk, repeat):
        rng = np.random.default_rng(rng_seed + repeat)
        own = features[name][chunk["event_row"].to_numpy()]
        drawn = rng.choice(pool, size=len(own))
        drawn = np.where(drawn == own, rng.choice(pool, size=len(own)), drawn)
        return interventions.set_feature(inputs, name, positions,
                                         torch.as_tensor(drawn, device=DEVICE)), None
    return make


def adjacent_edit(name, top):
    def make(inputs, positions, chunk, repeat):
        own = features[name][chunk["event_row"].to_numpy()]
        step = np.where(own >= top, -1, 1)
        return interventions.set_feature(inputs, name, positions,
                                         torch.as_tensor(np.clip(own + step, 0, top), device=DEVICE)), None
    return make


def shift_edit(name, sd):
    def make(inputs, positions, chunk, repeat):
        own = features[name][chunk["event_row"].to_numpy()].astype(np.float32)
        direction = 1.0 if repeat % 2 == 0 else -1.0
        return interventions.set_feature(inputs, name, positions,
                                         torch.as_tensor(np.clip(own + direction * sd, 0.0, 1.0 if name in ("x_norm", "y_norm") else (np.log1p(600.0) if name == "t_poss" else None)), device=DEVICE)), None
    return make


def noop_edit(inputs, positions, chunk, repeat):
    return inputs, None


def whole_context_edit(inputs, positions, chunk, repeat):
    donor_inputs, _ = mi_setup.window_inputs(bundle, val_view, chunk["donor_window"].to_numpy().astype(int))
    return interventions.write_identity(donor_inputs, inputs, positions, ["player"]), None


def random_vector_edit(group, seed):
    generator = torch.Generator().manual_seed(seed)
    def make(inputs, positions, chunk, repeat):
        return inputs, {f"factor_{group}": interventions.norm_matched_vector(None, positions, generator)}
    return make


with_donor = population["donor_window"].notna()
print("Legacy guessed adjacency and unbounded survey archived; execute corrected Section R below.")


Legacy guessed adjacency and unbounded survey archived; execute corrected Section R below.


> **⚠ REPAIR NOTE (2026-10-04) — VALID — INTERPRETATION NARROWED — F19, F20.** The magnitudes and intervals describe these interventions. The q column is the bootstrap floor for every row (an absolute change is never ≤ 0) and carries no ranking information. "[upper bound]" / "[floor]" labels are wrong (F19).

In [6]:
print("Obsolete historical survey section archived; corrected survey runs in Section R.")


Obsolete historical survey section archived; corrected survey runs in Section R.


In [7]:
print("Obsolete historical survey section archived; corrected survey runs in Section R.")


Obsolete historical survey section archived; corrected survey runs in Section R.


**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):

## 4. Group against the sum of its parts

Each whole factor group is ~~removed~~ *set to code 0 (F17: for context features code 0 is a learned category, so this is not removal)* at the target — every categorical input in it set to index 0 and
every continuous input to 0 — and compared with the sum of its members' individual resample
effects. ~~A group that moves the arbitration much more than its parts added together is being read
jointly; one that matches the sum is read input by input.~~ *[Corrected 2026-10-04 (F18): the group and its members use different endpoints, and |joint| − Σ|single| is negative even for an additive response whenever single effects have opposite signs. The table cannot establish additivity or interaction. Section R3 uses the same endpoints and signed contrasts.]* This is a coarse additivity check, not a
localisation.

> **⚠ REPAIR NOTE (2026-10-04) — INVALIDATED — EXPERIMENT DESIGN — F17, F18.** "removed" is set-to-code-0, and "group minus sum of parts" compares different endpoints with absolute values. Corrected: Section R3.

In [8]:
print("Obsolete historical survey section archived; corrected survey runs in Section R.")


Obsolete historical survey section archived; corrected survey runs in Section R.


## 5. Where in depth does each of the strongest inputs land?

Deliberately coarse: for the handful of inputs with the largest effect above, denoising activation
patching of the **residual stream after each block only** (4 cells per input), at the target. Not
the 21-site sweep of MB06 — just enough to say whether an input's effect is already present early
or appears late.

~~A flat profile that is high everywhere means the input's effect enters at the embedding and simply
rides the residual stream. A profile that climbs means later layers build something out of it.~~ *[Corrected 2026-10-04 (F21): the edit changes only the target position, and restoring the whole clean residual there restores everything downstream, so R = 1 at every depth is guaranteed. It is a restoration sanity check and cannot say where an input is processed.]*

> **⚠ REPAIR NOTE (2026-10-04) — VALID AS A SANITY CHECK ONLY — F21.** R = 1 at every depth is guaranteed by the design; no inference about processing depth.

In [9]:
print("Obsolete historical survey section archived; corrected survey runs in Section R.")


Obsolete historical survey section archived; corrected survey runs in Section R.


In [10]:
print("Obsolete historical survey section archived; corrected survey runs in Section R.")


Obsolete historical survey section archived; corrected survey runs in Section R.


## 6. Each factor group's direct write to the logits

The MB09 decomposition, restricted to this population and shown per factor group. Descriptive: a group that writes strongly towards one action is not thereby shown to be *used*, and a group that writes nothing directly may still act through later layers.

In [11]:
print("Obsolete historical survey section archived; corrected survey runs in Section R.")


Obsolete historical survey section archived; corrected survey runs in Section R.


**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):

In [12]:
print("Obsolete historical survey section archived; corrected survey runs in Section R.")


Obsolete historical survey section archived; corrected survey runs in Section R.


## 7. What this survey is for, and what it is not

**It is for** choosing which context input deserves a real experiment. The ranking in Section 3 and
the depth profile in Section 5 are enough to pick one or two, no more.

**It is not** evidence about the mechanism:

* Single-input edits measure a **main effect**. The open question from the audit is an
  **interaction** — whether a context input *gates identity* — and nothing here crosses a context
  edit with an identity edit. That needs the MB04 2×2 factorial rebuilt with a single context
  factor in place of the donor window.
* The inputs are correlated in the data (minute with score, zone with previous end zone), so a
  large effect for one does not mean the others are idle.
* Everything is on the validation estimation and discovery halves together, with no
  discovery/estimation separation, because nothing here is being selected for later confirmation.
  If an input from this table is promoted to a real experiment, that experiment has to do its own
  selection on the discovery half.
* Resample and adjacent edits are different sizes of change and are not comparable with each other,
  only within their own kind.

## R. The survey with interventions that match their names (REPAIR 2026-10-04)

| Edit (new name) | What it does | Replaces |
|---|---|---|
| **resample-other** | the field at the target gets a value drawn from the P-player population's values of that field **excluding the event's own**; the draw depends only on (seed, repeat, event_row), so it is batch-invariant; the realised unchanged share is reported | "resample" (F14, F15) |
| **ordinal step +1 / −1** | one step along the field's declared ordered codes, each direction reported separately; special codes never edited or produced; rows at a boundary skipped | "adjacent" on ordered fields (F16) |
| **grid step (row / col, ±1)** | the spatial bin's grid neighbour under the **declared** zone layout | "adjacent" on zones (F16) |
| *(none)* | unordered categories (prev_action, prev_outcome, possession_flag, period, same_player_as_prev) get **no** adjacent edit | — |
| **±1 SD (transformed units)** | as before, labelled by the stored feature's units (dt and t_poss are stored transformed) | "shift" |
| **[reference] whole-window donor swap** | as before | "upper bound" (F19) |
| **[reference] norm-matched random factor vector** | as before | "floor" (F19) |
| **set-to-code-0 (group)** | as Section 4, literally named | "group removed" (F17) |
| **group factor mean-ablation** | the group's factor vector at the target replaced by its mean over the population's targets: an activation-level ablation | — (F17) |
| **signed non-additivity** | single-field and joint edits to the **same donor event's values**; per event joint − Σ single, signed | "group minus sum" (F18) |

Effects are reported as mean |ΔΔ| with match-bootstrap intervals, the share of rows actually changed, and the effect as a
share of the whole-window reference. **No significance column** (F20): an |ΔΔ| > 0 test only says an edit is not a no-op.

**Ordinal and spatial semantics must be confirmed (F16).** Which codes are ordered, which are special, and how zone codes map
to a grid are properties of `model_training_evaluation/src/data.py`. The repair did not have that file, so the cell below
prints what the data module exposes, and the ordinal and grid edits run **only after** the researcher has filled in and confirmed
`ORDINAL_CODES` and `ZONE_LAYOUT`. Everything else runs regardless.

In [13]:
import importlib
try:
    import mi_src.scrub as scrub
    import mi_src.steering as steering
    import mi_src.context_edits as context_edits
    import mi_src.graph as graph
except ModuleNotFoundError as error:
    raise RuntimeError("BLOCKED: the repaired modules (scrub, steering, context_edits, graph) are missing. Run "
                       "MB07R_repaired_sources_and_regression_tests.ipynb, then restart the kernel.") from error
for module in [hooks, mi_metrics, circuits, scrub, steering, context_edits, graph]:
    importlib.reload(module)
for module in [circuits, mi_metrics, scrub, steering, context_edits, graph]:
    if getattr(module, "REPAIR_VERSION", None) != "2026-10-04":
        raise RuntimeError(f"BLOCKED: {module.__name__} on disk is not the repaired 2026-10-04 version. Re-run the repaired "
                           "%%writefile cells (MB02 for mi_metrics, MB07 for circuits, MB07R for the new modules), then restart.")
print("repaired modules loaded:", [m.__name__ for m in [circuits, mi_metrics, scrub, steering, context_edits, graph]])

print("data-module names that may define encodings:", [n for n in dir(data) if any(k in n.upper() for k in ["ZONE", "BIN", "ACTION", "OUTCOME", "CODE", "SCORE", "POSS"])])
SPECIAL_PREV_ACTION = set()
if hasattr(data, "END_ACTION_CODE"):
    SPECIAL_PREV_ACTION.add(int(data.END_ACTION_CODE))
if "OTHER" in list(data.ACTION_NAMES):
    SPECIAL_PREV_ACTION.add(list(data.ACTION_NAMES).index("OTHER"))
print("special prev_action codes (never edited or drawn):", sorted(SPECIAL_PREV_ACTION))

# --- TO BE FILLED IN AND CONFIRMED BY THE RESEARCHER from model_training_evaluation/src/data.py (F16) ---
ORDINAL_CODES = {        # field -> list of codes in their true order, special codes excluded
    "minute_bin": None,  # e.g. list(range(n_regular_bins)) if bins are consecutive minutes; exclude any extra-time / missing code
    "score_diff": None,  # e.g. the codes of -k..+k in order
    "poss_len": None,    # e.g. ordered length bins
}
ZONE_LAYOUT = None       # e.g. {"n_rows": .., "n_cols": .., "cell_of_code": callable, "code_of_cell": callable}
ADJACENCY_CONFIRMED = False   # set True only after checking every entry above against data.py
# -----------------------------------------------------------------------------------------------------------

coverage = pd.DataFrame([{"population": name, "rows": int((val_conflicts.population == name).sum())} for name in ["P-player", "index-0"]]
                        + [{"population": "P-player with a donor window (whole-window reference)", "rows": int(population["donor_window"].notna().sum())}])
print(coverage.to_string(index=False))
print("All of Section R uses the P-player rows only; nothing here is generalised to index-0 rows or to non-conflict events (F11, F12).")

repaired modules loaded: ['mi_src.circuits', 'mi_src.mi_metrics', 'mi_src.scrub', 'mi_src.steering', 'mi_src.context_edits', 'mi_src.graph']
data-module names that may define encodings: ['ACTION_NAMES', 'END_ACTION_CODE', 'NUM_ACTIONS', 'POSSESSION_SOURCE_COLUMNS', 'add_pre_action_possession_features', 'encode_features']
special prev_action codes (never edited or drawn): [14, 15]
                                           population  rows
                                             P-player  1744
                                              index-0  8947
P-player with a donor window (whole-window reference)  1743
All of Section R uses the P-player rows only; nothing here is generalised to index-0 rows or to non-conflict events (F11, F12).


In [14]:
# R2. Single-input edits, corrected
EVENT_ROWS = population["event_row"].to_numpy()
pool_of = {name: features[name][EVENT_ROWS] for _, name in CATEGORICAL + CONTINUOUS}
noop_rows, frames_r, changed_share = [], [], {}


def resample_other_edit(name, seed):
    pool = pool_of[name]
    if name == "prev_action" and SPECIAL_PREV_ACTION:
        pool = pool[~np.isin(pool, sorted(SPECIAL_PREV_ACTION))]
    def make(inputs, positions, chunk, repeat):
        rows = chunk["event_row"].to_numpy()
        own = features[name][rows]
        new, unchanged = context_edits.resample_other(pool, own, rows, seed, repeat)
        if name == "prev_action" and SPECIAL_PREV_ACTION:
            new = np.where(np.isin(own, sorted(SPECIAL_PREV_ACTION)), own, new)
        return interventions.set_feature(inputs, name, positions, torch.as_tensor(new, device=DEVICE)), None
    return make


def changed_fraction(name, seed):
    pool = pool_of[name]
    if name == "prev_action" and SPECIAL_PREV_ACTION:
        pool = pool[~np.isin(pool, sorted(SPECIAL_PREV_ACTION))]
    shares = []
    for repeat in range(DRAWS):
        own = features[name][EVENT_ROWS]
        new, _ = context_edits.resample_other(pool, own, EVENT_ROWS, seed, repeat)
        if name == "prev_action" and SPECIAL_PREV_ACTION:
            new = np.where(np.isin(own, sorted(SPECIAL_PREV_ACTION)), own, new)
        shares.append(np.mean(new != own))
    return float(np.mean(shares))


start = time.time()
frames_r.append(measure("no-op", noop_edit, [], 1))
for position, (group, name) in enumerate(CATEGORICAL + CONTINUOUS):
    seed = 1000 + position
    frames_r.append(measure(f"{name} resample-other", resample_other_edit(name, seed), [name], DRAWS))
    changed_share[f"{name} resample-other"] = changed_fraction(name, seed)
    noop_rows.append({"input": name, "original rule: expected unchanged share": context_edits.two_draw_noop_rate(pool_of[name], pool_of[name])
                      if (group, name) in CATEGORICAL else np.nan,
                      "corrected rule: realised unchanged share": 1 - changed_share[f"{name} resample-other"],
                      "of which: rows whose own code is special (never edited)": float(np.isin(pool_of[name], sorted(SPECIAL_PREV_ACTION)).mean())
                      if name == "prev_action" and SPECIAL_PREV_ACTION else 0.0})
for group, name in CONTINUOUS:
    sd = float(pool_of[name].astype(np.float64).std())
    frames_r.append(measure(f"{name} +/- 1 SD (bounded stored units)", shift_edit(name, sd), [name], 2))
    changed_share[f"{name} +/- 1 SD (bounded stored units)"] = float(np.mean([np.mean(context_edits.bounded_shift(pool_of[name], sd, direction, upper=1.0 if name in ("x_norm", "y_norm") else (np.log1p(600.0) if name == "t_poss" else None))[1]["changed"]) for direction in (-1, 1)]))
if ADJACENCY_CONFIRMED:
    for name, codes in ORDINAL_CODES.items():
        if codes is None or name not in features:
            continue
        for direction in (+1, -1):
            new, moved = context_edits.ordinal_step(pool_of[name], codes, direction)
            lookup = dict(zip(EVENT_ROWS, new))
            label = f"{name} ordinal step {direction:+d}"
            frame = measure(label, lambda inputs, positions, chunk, repeat, n=name, lk=lookup: (
                interventions.set_feature(inputs, n, positions, torch.as_tensor([lk[r] for r in chunk["event_row"].to_numpy()], device=DEVICE)), None), [name], 1)
            frames_r.append(frame[moved].reset_index(drop=True))
            changed_share[label] = float(moved.mean())
    if ZONE_LAYOUT is not None:
        for name in [n for n in ["zone", "prev_end_zone"] if n in features]:
            for axis, direction in [("row", 1), ("row", -1), ("col", 1), ("col", -1)]:
                new, moved = context_edits.grid_step(pool_of[name], ZONE_LAYOUT["n_rows"], ZONE_LAYOUT["n_cols"], axis, direction,
                                                     ZONE_LAYOUT["code_of_cell"], ZONE_LAYOUT["cell_of_code"])
                lookup = dict(zip(EVENT_ROWS, new))
                label = f"{name} grid step {axis} {direction:+d}"
                frame = measure(label, lambda inputs, positions, chunk, repeat, n=name, lk=lookup: (
                    interventions.set_feature(inputs, n, positions, torch.as_tensor([lk[r] for r in chunk["event_row"].to_numpy()], device=DEVICE)), None), [name], 1)
                frames_r.append(frame[moved].reset_index(drop=True))
                changed_share[label] = float(moved.mean())
else:
    print("ordinal / grid steps skipped: ADJACENCY_CONFIRMED is False (fill in ORDINAL_CODES / ZONE_LAYOUT from data.py first)")
for group in CONTEXT_GROUPS:
    frames_r.append(measure(f"[reference] norm-matched random factor vector, {group}", random_vector_edit(group, 7), None, DRAWS))
saved_population = population
population = population[with_donor].reset_index(drop=True)
frames_r.append(measure("[reference] whole-window donor swap (player written back)", whole_context_edit, None, 1))
population = saved_population
effects_r = pd.concat(frames_r, ignore_index=True)
assert float(effects_r.loc[effects_r.edit == "no-op", "abs_delta_change"].max()) == 0.0
print(f"corrected single-input edits done in {time.time() - start:.0f} s")

reference_size = float(effects_r.loc[effects_r.edit.str.startswith("[reference] whole-window"), "abs_delta_change"].mean())
rows_r = []
for edit, group in effects_r.groupby("edit", sort=False):
    stats = mi_metrics.summarise(group["abs_delta_change"], group["match_id"], N_BOOT)
    rows_r.append({"edit": edit, "n rows scored": len(group), "share of rows actually changed": changed_share.get(edit, np.nan),
                   "mean |change in delta|": stats["mean"], "CI low": stats["ci_low"], "CI high": stats["ci_high"],
                   "as a share of the whole-window reference": stats["mean"] / reference_size if reference_size else np.nan,
                   "mean KL": float(group["kl"].mean()), "mean change in log p(a_ctx)": float(group["dlogp_ctx"].mean()),
                   "mean change in log p(a_id)": float(group["dlogp_id"].mean())})
summary_r = pd.DataFrame(rows_r).sort_values("mean |change in delta|", ascending=False)
noop_table = pd.DataFrame(noop_rows)
summary_r.to_csv(OUT / "results" / "MB13_single_inputs_corrected.csv", index=False)
noop_table.to_csv(OUT / "results" / "MB13_noop_rates_corrected.csv", index=False)
print(summary_r.round(4).to_string(index=False))
print("\nunchanged share per field: original two-draw rule (expected) vs corrected rule (realised):")
print(noop_table.round(4).to_string(index=False))
shift_coverage = []
for _, name in CONTINUOUS:
    upper = 1.0 if name in ("x_norm", "y_norm") else (np.log1p(600.0) if name == "t_poss" else None)
    for direction in (-1, 1):
        values, info = context_edits.bounded_shift(pool_of[name], float(pool_of[name].astype(float).std()), direction, upper=upper)
        assert (values >= 0).all() and (upper is None or (values <= upper + 1e-6).all())
        shift_coverage.append({"feature": name, "direction": direction, "clipped share": float(info["clipped"].mean()), "changed share": float(info["changed"].mean()), "mean realized absolute dose (stored units)": float(info["absolute_dose"].mean())})
pd.DataFrame(shift_coverage).to_csv(OUT / "results/MB13_shift_coverage.csv", index=False)
print(pd.DataFrame(shift_coverage).to_string(index=False))


ordinal / grid steps skipped: ADJACENCY_CONFIRMED is False (fill in ORDINAL_CODES / ZONE_LAYOUT from data.py first)


corrected single-input edits done in 18 s
                                                     edit  n rows scored  share of rows actually changed  mean |change in delta|  CI low  CI high  as a share of the whole-window reference  mean KL  mean change in log p(a_ctx)  mean change in log p(a_id)
[reference] whole-window donor swap (player written back)           1743                             NaN                  6.7951  6.4547   7.1452                                    1.0000   5.9525                      -6.1220                      0.3734
                                        dt resample-other           1744                          1.0000                  5.8946  5.4805   6.2510                                    0.8675   2.9083                      -3.0086                      2.5381
   [reference] norm-matched random factor vector, history           1744                             NaN                  5.1601  4.8008   5.5231                                    0.7594   2.3473

### R3 — Groups: literal set-to-code-0, a genuine activation-level ablation, and signed non-additivity (F17, F18)

* **set-to-code-0** repeats Section 4's operation under its literal name.
* **group factor mean-ablation** replaces `factor_<group>` at the target by its mean over the population's targets. This is
  what "the group's contribution removed" means at the activation level.
* **signed non-additivity.** One donor event per event (another P-player event, fixed by (seed, event_row)). Each field of
  the group is set to the donor's value alone (single), then all of them together (joint). Every change is signed, from the
  same baseline to the same endpoint. Per event, joint − Σ single is exactly 0 if the response is additive in those fields.
  Its mean, interval and mean absolute size relative to mean |joint| are reported. The edited events remain unseen joint
  combinations within the window (the fields are copied together from one donor, but the rest of the window is not).

In [15]:
@torch.no_grad()
def signed_changes(make_inputs, scope):
    out = np.empty(len(population))
    for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, population, BATCH):
        chunk = population.iloc[part]
        base = circuits.delta_under(MODEL, inputs, positions, a_ctx, a_id)
        changed, replace = make_inputs(inputs, positions, chunk)
        if scope is not None:
            interventions.assert_scope(inputs, changed, scope, positions)
        out[part] = circuits.delta_under(MODEL, changed, positions, a_ctx, a_id, replace) - base
    return out


factor_means = {}
with torch.no_grad():
    sums, count = {}, 0
    for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, population, BATCH):
        _, cache = hooks.forward(MODEL, inputs, capture=[f"factor_{g}" for g in CONTEXT_GROUPS])
        rows = torch.arange(len(positions), device=DEVICE)
        for g in CONTEXT_GROUPS:
            value = cache[f"factor_{g}"][rows, positions].sum(0)
            sums[g] = value if g not in sums else sums[g] + value
        count += len(positions)
    factor_means = {g: sums[g] / count for g in CONTEXT_GROUPS}

donors_r = context_edits.donor_rows_for(EVENT_ROWS, EVENT_ROWS, seed=77)
group_rows = []
for group in CONTEXT_GROUPS:
    names = [n for n in FACTOR_GROUPS[group] if n in features]
    continuous = [n for n in CONTINUOUS_GROUPS.get(group, []) if n in features]
    fields = names + continuous
    matches = population["match_id"].to_numpy()
    code_zero = signed_changes(lambda inputs, positions, chunk: (context_edits.set_to_code_zero(inputs, names, positions, continuous), None), fields)
    mean_ablation = signed_changes(lambda inputs, positions, chunk, g=group: (inputs, {f"factor_{g}": context_edits.factor_mean_ablation(factor_means[g], positions)}), None)
    donor_values = context_edits.joint_donor_values(features, donors_r, fields)
    lookup = {name: dict(zip(EVENT_ROWS, values)) for name, values in donor_values.items()}
    def endpoint(subset):
        return lambda inputs, positions, chunk: (context_edits.apply_values(inputs, positions,
                    {n: [lookup[n][r] for r in chunk["event_row"].to_numpy()] for n in subset}), None)
    singles = [signed_changes(endpoint([n]), [n]) for n in fields]
    joint = signed_changes(endpoint(fields), fields)
    nonadditive = context_edits.signed_nonadditivity(joint, singles)
    old_statistic = float(np.abs(joint).mean() - np.sum([np.abs(s).mean() for s in singles]))
    for label, values in [("set-to-code-0 (literal)", code_zero), ("group factor mean-ablation", mean_ablation),
                          ("joint donor edit (same endpoint as singles)", joint)]:
        stats = mi_metrics.summarise(np.abs(values), matches, N_BOOT)
        group_rows.append({"group": group, "quantity": f"|change| under {label}", "mean": stats["mean"], "CI low": stats["ci_low"], "CI high": stats["ci_high"]})
    stats = mi_metrics.summarise(nonadditive, matches, N_BOOT)
    group_rows.append({"group": group, "quantity": "signed non-additivity: joint - sum(single), mean", "mean": stats["mean"], "CI low": stats["ci_low"], "CI high": stats["ci_high"]})
    stats = mi_metrics.summarise(np.abs(nonadditive), matches, N_BOOT)
    group_rows.append({"group": group, "quantity": "|joint - sum(single)|, mean", "mean": stats["mean"], "CI low": stats["ci_low"], "CI high": stats["ci_high"]})
    group_rows.append({"group": group, "quantity": "|joint - sum(single)| / |joint| (ratio of means)", "mean": float(np.abs(nonadditive).mean() / max(np.abs(joint).mean(), 1e-12)),
                       "CI low": np.nan, "CI high": np.nan})
    group_rows.append({"group": group, "quantity": "for reference: original statistic on these endpoints, |joint| - sum|single|", "mean": old_statistic,
                       "CI low": np.nan, "CI high": np.nan})
groups_r = pd.DataFrame(group_rows)
groups_r.to_csv(OUT / "results" / "MB13_groups_corrected.csv", index=False)
print(groups_r.round(4).to_string(index=False))
mi_setup.save_json(OUT / "results" / "MB13_summary_corrected.json", {
    "status": "REPAIRED 2026-10-04 (F14-F21); side study; not pre-registered; no discovery/estimation split",
    "adjacency_confirmed": ADJACENCY_CONFIRMED, "special_prev_action_codes": sorted(SPECIAL_PREV_ACTION),
    "single_inputs": summary_r.to_dict(orient="records"), "noop_rates": noop_table.to_dict(orient="records"),
    "groups": groups_r.to_dict(orient="records"), "run_mode": RUN_MODE})
fingerprint.record_files(MANIFEST, {"results/MB13_summary_corrected.json": OUT / "results" / "MB13_summary_corrected.json"})
fingerprint.register(REGISTRY, {"notebook": "MB13", "experiment": "R: corrected context survey (F14-F21)", "seed": PRIMARY_SEED, "run_mode": RUN_MODE})

  group                                                                    quantity    mean  CI low  CI high
  space                                      |change| under set-to-code-0 (literal)  1.1141  1.0443   1.1803
  space                                   |change| under group factor mean-ablation  0.9986  0.9156   1.0862
  space                  |change| under joint donor edit (same endpoint as singles)  1.1618  1.0662   1.2610
  space                            signed non-additivity: joint - sum(single), mean -0.1242 -0.1363  -0.1101
  space                                                 |joint - sum(single)|, mean  0.2253  0.2105   0.2395
  space                            |joint - sum(single)| / |joint| (ratio of means)  0.1940     NaN      NaN
  space for reference: original statistic on these endpoints, |joint| - sum|single| -0.1496     NaN      NaN
   time                                      |change| under set-to-code-0 (literal)  1.8787  1.6091   2.2385
   time            

**What Section R can establish.** For each context field, how much the arbitration moves under an edit that actually changes the
field, where "other value" excludes the original and the unchanged share is measured. Whether each group's response to a
coherent single-donor change is additive in its fields (signed, same endpoints). How much the group's activation-level
contribution matters (mean-ablation).

**What it cannot establish.** Which context variable **gates identity**: nothing here crosses a context edit with an identity
edit. That still needs the MB04 2×2 rebuilt with one context factor, with selection on the discovery half and estimation on the
other (L09). The edited events are still not guaranteed to be realistic football states. Depth (Section 5) remains a sanity check.

**Researcher interpretation of Section R** *(to be written after reading the corrected numbers)*

- What was tested:
- What was observed:
- What it supports / contradicts:
- What it does not establish:

R15–R16: continuous shifts respect x/y [0,1], nonnegative transformed gaps, and the source-defined t_poss cap log(601). Boundary clipping and actual dose are reported. Marginal edits remain factor sensitivity probes and can break cross-field dependencies; their effects are not a comparable real-world importance ranking. Spatial joint donor edits in the group table preserve the observed zone/x/y tuple, while no claim is made about consistency with the rest of the window. Codebook-dependent adjacency remains skipped until source-confirmed semantics are available.